# Dados para atualizar o dashboard


In [4]:
from pyspark.sql import SparkSession


spark = (
    SparkSession
    .builder
    .appName("Agendamentos")
    .config("spark.jars", "/home/jovyan/work/jars/hadoop-aws-3.3.4.jar,/home/jovyan/work/jars/aws-java-sdk-bundle-1.12.262.jar")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "miniominio")
    .config("spark.hadoop.fs.s3a.secret.key", "miniominio")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)


In [6]:
data_inicial = "20/04/2026"
data_final = "19/05/2026"

data_inicial_read = data_inicial.replace("/", "-")
data_final_read = data_final.replace("/", "-")

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(f"s3a://agendamentos/gold/agendamentos_{data_inicial_read}_{data_final_read}")
)

df.createOrReplaceTempView("agendamentos")


In [22]:

valores_procedimentos = {
    "PACOTE PRÉ-OPERATÓRIO ADULTO - PROCTOLOGISTA": 450.00,
    "PACOTE PRÉ-OPERATÓRIO ADULTO - GINECOLOGIA": 450.00,
    "PACOTE PRÉ-OPERATÓRIO ADULTO - UROLOGIA": 450.00,
    "PACOTE PRÉ-OPERATÓRIO ADULTO - VASCULAR": 450.00,
    "PACOTE PRÉ-OPERATÓRIO ADULTO CIRURGIA GERAL": 450.00,
    "PACOTE RISCO CIRURGICO - ADULTO": 400.00,
    "CIRURGIA DE VARIZES UNILATERAL (COM SAFENA) - ADULTO": 8132.00,
    "CIRURGIA DE HEMORROIDECTOMIA - ADULTO": 5850.00,
    "CIRURGIA DE COLECISTECTOMIA - ADULTO": 10362.30,
    "CIRURGIA DE COLECISTECTOMIA VIDEOLAPAROSCÓPICA - ADULTO": 13100.00,
    "CIRURGIA DE HERNIOPLASTIA EPIGASTRICA - ADULTO": 5850.00,
    "CIRURGIA DE HERNIOPLASTIA INCISIONAL - ADULTO": 6175.00,
    "CIRURGIA DE HERNIOPLASTIA INGUINAL (UNILATERAL) - ADULTO": 5850.00,
    "CIRURGIA DE HERNIOPLASTIA RECIDIVANTE - ADULTO": 7312.50,
    "CIRURGIA DE HERNIOPLASTIA UMBILICAL - ADULTO": 6175.00,
    "CIRURGIA DE HERNIORRAFIA INGUINAL VIDEOLAPAROSCÓPICA - ADULTO": 8626.00,
    "CIRURGIA DE HERNIORRAFIA UMBILICAL VIDEOLAPAROSCÓPICA - ADULTO": 8626.00,
    "CIRURGIA DE LAPAROTOMIA EXPLORADORA - ADULTO": 10838.23,
    "CIRURGIA DE INSTALAÇÃO ENDOSCÓPICA DE CATETER DUPLO J - ADULTO": 5173.36,
    "CIRURGIA DE LITOTRIPSIA - ADULTO": 13750.00,
    "CIRURGIA DE URETROPLASTIA (RESSECÇÃO DE CORDA) - ADULTO": 13500.00,
    "CIRURGIA DE RESSECÇÃO ENDOSCÓPICA DE PRÓSTATA - ADULTO": 14500.00,
    "CIRURGIA DE ESPERMATOCELECTOMIA - ADULTO": 4850.00,
    "CIRURGIA DE HIDROCELE - ADULTO": 4850.00,
    "CIRURGIA DE VARICOCELE (VARICOCELECTOMIA) - ADULTO": 5082.80,
    "CIRURGIA DE VASECTOMIA - ADULTO": 5850.00,
    "CIRURGIA DE PLASTICA TOTAL DO PENIS - ADULTO": 12500.00,
    "CIRURGIA DE POSTECTOMIA (FIMOSE) - ADULTO": 5850.00,
    "CIRURGIA DE CURETAGEM SEMIOTICA - ADULTO": 5850.00,
    "CIRURGIA DE HISTERECTOMIA TOTAL - ADULTO": 10886.20,
    "CIRURGIA DE HISTERECTOMIA VIDEOLAPAROSCOPICA - ADULTO": 16340.25,
    "CIRURGIA DE LAQUEADURA TUBARIA - ADULTO": 6500.00,
    "CIRURGIA DE MIOMECTOMIA - ADULTO": 8500.00,
    "CIRURGIA DE MIOMECTOMIA VIDEOLAPAROSCOPICA - ADULTO": 16125.00,
    "CIRURGIA DE OOFORECTOMIA / OOFOROPLASTIA - ADULTO": 8500.00,
    "CIRURGIA DE SALPINGECTOMIA UNI / BILATERAL - ADULTO": 8500.00,
    "CIRURGIA DE COLPOPERINEOPLASTIA ANTERIOR E POSTERIOR - ADULTO": 16500.00,
    "CIRURGIA DE EXERESE DE CISTO VAGINAL - ADULTO": 4000.00,
    "CIRURGIA DE EXERESE DE GLÂNDULA DE BARTHOLIN / SKENE - ADULTO": 3500.00,
    "CIRURGIA DE PROSTATECTOMIA - ADULTO": 8385.29
}


In [ ]:
from pyspark.sql import functions as F

# Transforma o dicionário em um mapa Spark
mapa_precos = F.create_map(
    *[
        item
        for nome, valor in valores_procedimentos.items()
        for item in (F.lit(nome), F.lit(valor))
    ]
)

# Consulta agrupada: somente agendamentos validados
df_resultado = spark.sql("""
    SELECT
        opera,
        prestador,
        procedimento,
        cidade,
        emenda,
        competencia,
        COUNT(*) AS quantidade
    FROM agendamentos
    WHERE situacao = 'VALIDADO'
    GROUP BY
        opera,
        prestador,
        procedimento,
        cidade,
        emenda,
        competencia
""")

# Busca o valor unitário pelo nome exato do procedimento
df_resultado = df_resultado.withColumn(
    "valor_unitario",
    F.element_at(
        mapa_precos,
        F.col("procedimento")
    ).cast("decimal(14,2)")
)

# Calcula o total
df_resultado = df_resultado.withColumn(
    "total",
    (
        F.col("quantidade") * F.col("valor_unitario")
    ).cast("decimal(14,2)")
)

# Formata os valores:
# 1200.35 -> 1200,35
# 13100.00 -> 13100
# 450.00 -> 450
def formatar_valor(coluna):
    valor = F.col(coluna).cast("decimal(14,2)")
    texto = valor.cast("string")

    return (
        F.when(
            valor.isNull(),
            F.lit(None).cast("string")
        )
        .when(
            valor == F.floor(valor),
            F.regexp_replace(texto, r"\.00$", "")
        )
        .otherwise(
            F.regexp_replace(texto, r"\.", ",")
        )
    )

# Aplica a formatação e organiza as colunas
df_resultado = (
    df_resultado
    .withColumn("valor_unitario", formatar_valor("valor_unitario"))
    .withColumn("total", formatar_valor("total"))
    .select(
        "opera",
        "prestador",
        "procedimento",
        "valor_unitario",
        "quantidade",
        "cidade",
        "emenda",
        "competencia",
        "total"
    )
    .orderBy(
        "cidade",
        "prestador",
        "procedimento"
    )
)

# Converte para Pandas e exibe o resultado
df_pandas = df_resultado.toPandas()

df_pandas

,opera,prestador,procedimento,valor_unitario,quantidade,cidade,emenda,competencia,total
0,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,CIRURGIA DE COLECISTECTOMIA VIDEOLAPAROSCÓPICA...,13100,15,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,196500
1,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,CIRURGIA DE HERNIORRAFIA INGUINAL VIDEOLAPAROS...,8626,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,8626
2,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,CIRURGIA DE HERNIORRAFIA UMBILICAL VIDEOLAPARO...,8626,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,8626
3,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,CIRURGIA DE VASECTOMIA - ADULTO,5850,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,5850
4,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,PACOTE PRÉ-OPERATÓRIO ADULTO - UROLOGIA,450,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,450
5,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,PACOTE PRÉ-OPERATÓRIO ADULTO CIRURGIA GERAL,450,20,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,9000
6,Opera Adulto,DESAM - HOSPITAL LUZIA BENEVENUTO,PACOTE RISCO CIRURGICO - ADULTO,400,18,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,7200
7,Opera Adulto,Hospital Clinico Cirurgico Catarina Ltda,CIRURGIA DE HERNIORRAFIA UMBILICAL VIDEOLAPARO...,8626,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,8626
8,Opera Adulto,Hospital Clinico Cirurgico Catarina Ltda,PACOTE PRÉ-OPERATÓRIO ADULTO CIRURGIA GERAL,450,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,450
9,Opera Adulto,Hospital Clinico Cirurgico Catarina Ltda,PACOTE RISCO CIRURGICO - ADULTO,400,1,Belford Roxo,Opera Adulto (SES),20/04 a 19/05,400


## Pacotes

In [ ]:


df_resultado = spark.sql("""
 WITH dados AS (
    SELECT
        CASE
            WHEN procedimento IN (
                'PACOTE PRÉ-OPERATÓRIO ADULTO - GINECOLOGIA'
            ) 
            THEN 'PACOTE GINECOLOGISTA'
           
            WHEN procedimento IN (
                'PACOTE PRÉ-OPERATÓRIO ADULTO - PROCTOLOGISTA'
            ) 
            THEN 'PACOTE PROCTOLOGISTA'
           
            WHEN procedimento IN (
                'PACOTE PRÉ-OPERATÓRIO ADULTO - UROLOGIA'
            ) 
            THEN 'PACOTE UROLOGISTA'
           
            WHEN procedimento IN (
                'PACOTE PRÉ-OPERATÓRIO ADULTO - VASCULAR'
            ) 
            THEN 'PACOTE VASCULAR'
           
            WHEN procedimento IN (
                'PACOTE PRÉ-OPERATÓRIO ADULTO CIRURGIA GERAL'
            ) 
            THEN 'PACOTE GERAL'

            WHEN procedimento IN (
                'PACOTE RISCO CIRURGICO - ADULTO'
            ) 
            THEN 'PACOTE RISCO'
            
            else 'outros'
        END AS procedimento
    FROM opera_dado
    WHERE procedimento like '%PACOTE%'
)
 
select procedimento
    ,count(*)
from dados
group by 1

""")

df_pandas = df_resultado.toPandas()

df_pandas

# cirurgias

In [ ]:

df_resultado = spark.sql(""" WITH dados AS (
    SELECT
        CASE
            WHEN procedimento IN (
                'CIRURGIA DE COLECISTECTOMIA - ADULTO',
                'CIRURGIA DE COLECISTECTOMIA VIDEOLAPAROSCÓPICA - ADULTO',
                'CIRURGIA DE COLPOPERINEOPLASTIA ANTERIOR E POSTERIOR - ADULTO',
                'CIRURGIA DE HERNIOPLASTIA EPIGASTRICA - ADULTO',
                'CIRURGIA DE HERNIOPLASTIA INCISIONAL - ADULTO',
                'CIRURGIA DE HERNIOPLASTIA INGUINAL (UNILATERAL) - ADULTO',
                'CIRURGIA DE HERNIOPLASTIA RECIDIVANTE - ADULTO',
                'CIRURGIA DE HERNIOPLASTIA UMBILICAL - ADULTO',
                'CIRURGIA DE HERNIORRAFIA INGUINAL VIDEOLAPAROSCÓPICA - ADULTO',
                'CIRURGIA DE HERNIORRAFIA UMBILICAL VIDEOLAPAROSCÓPICA - ADULTO'
            ) THEN 'Geral'

            WHEN procedimento IN (
                'CIRURGIA DE HEMORROIDECTOMIA - ADULTO'
            ) THEN 'Proctologista'

            WHEN procedimento IN (
                'CIRURGIA DE CURETAGEM SEMIOTICA - ADULTO',
                'CIRURGIA DE EXERESE DE CISTO VAGINAL - ADULTO',
                'CIRURGIA DE EXERESE DE GLÂNDULA DE BARTHOLIN / SKENE - ADULTO',
                'CIRURGIA DE HISTERECTOMIA TOTAL - ADULTO',
                'CIRURGIA DE HISTERECTOMIA VIDEOLAPAROSCOPICA - ADULTO',
                'CIRURGIA DE LAPAROTOMIA EXPLORADORA - ADULTO',
                'CIRURGIA DE LAQUEADURA TUBARIA - ADULTO',
                'CIRURGIA DE MIOMECTOMIA - ADULTO',
                'CIRURGIA DE MIOMECTOMIA VIDEOLAPAROSCOPICA - ADULTO',
                'CIRURGIA DE OOFORECTOMIA / OOFOROPLASTIA - ADULTO',
                'CIRURGIA DE SALPINGECTOMIA UNI / BILATERAL - ADULTO'
            ) THEN 'Ginecologista'

            WHEN procedimento IN (
                'CIRURGIA DE ESPERMATOCELECTOMIA - ADULTO',
                'CIRURGIA DE HIDROCELE - ADULTO',
                'CIRURGIA DE INSTALAÇÃO ENDOSCÓPICA DE CATETER DUPLO J - ADULTO',
                'CIRURGIA DE LITOTRIPSIA - ADULTO',
                'CIRURGIA DE PLASTICA TOTAL DO PENIS - ADULTO',
                'CIRURGIA DE POSTECTOMIA (FIMOSE) - ADULTO',
                'CIRURGIA DE PROSTATECTOMIA - ADULTO',
                'CIRURGIA DE RESSECÇÃO ENDOSCÓPICA DE PRÓSTATA - ADULTO',
                'CIRURGIA DE URETROPLASTIA (RESSECÇÃO DE CORDA) - ADULTO',
                'CIRURGIA DE VARICOCELE (VARICOCELECTOMIA) - ADULTO',
                'CIRURGIA DE VASECTOMIA - ADULTO'
            ) THEN 'Urologia'

            WHEN procedimento IN (
                'CIRURGIA DE VARIZES UNILATERAL (COM SAFENA) - ADULTO'
            ) THEN 'Vascular'

            ELSE 'Outros'
        END AS procedimento
    FROM opera_dado
    WHERE procedimento like 'CIRURGIA%'
)
 
select procedimento
    ,count(*)
from dados
group by 1

""")
df_pandas = df_resultado.toPandas()

df_pandas

# Consolidado

In [ ]:
df_resultado = spark.sql("""
    WITH resumo AS (
        SELECT
            CASE
                WHEN procedimento LIKE 'CIRURGIA%' THEN 'CIRURGIAS'
                WHEN procedimento LIKE '%PACOTE%' THEN 'Consulta'
            END AS categoria,
            COUNT(*) AS quantidade,
            emenda
        FROM opera_dado
        GROUP BY 1, emenda
    )

    SELECT emenda, categoria, quantidade
    FROM resumo
    ORDER BY categoria;
""")

df_pandas = df_resultado.toPandas()

df_pandas

# Faltas


In [ ]:
df_resultado = spark.sql("""

    select 
        emenda,
        prestador, 
        cidade, 
        competencia, 
        SUM(CASE WHEN situacao = 'FALTA' AND procedimento like '%PACOTE%' THEN 1 ELSE 0 END) as faltas_pacote,
        SUM(CASE WHEN situacao = 'VALIDADO' AND procedimento like '%PACOTE%' THEN 1 ELSE 0 END) as realizado_pacote,
        
        SUM(CASE WHEN situacao = 'FALTA' AND procedimento like 'CIRURGIA%' THEN 1 ELSE 0 END) as faltas_cirurgia,
        SUM(CASE WHEN situacao = 'VALIDADO' AND procedimento like 'CIRURGIA%' THEN 1 ELSE 0 END) as realizado_cirurgia,
        0 as nao_cirurgico,
        0 as reagendamento_cirurgia
        
    from opera_dado
    where situacao IN ('FALTA', 'VALIDADO')
    group by 
        prestador, 
        cidade, 
        competencia,
        emenda
    order by prestador, cidade
""")

df_pandas = df_resultado.toPandas()

df_pandas